In [1]:
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "4" 

In [2]:
import pandas as pd
import numpy as np
import csv
import sys
import gzip
import os

# ==========================================
# 0. 檔案路徑與參數設定
# ==========================================
# 請將這裡改成你電腦中檔案的實際路徑
PATH_ICUSTAYS = 'icustays.csv'
PATH_ADMISSIONS = 'admissions.csv'
PATH_PATIENTS = 'patients.csv'
PATH_PROCEDURES = 'procedureevents.csv' 
PATH_RADIOLOGY = 'radiology.csv.gz'     # 支援 .csv 或 .csv.gz

# 增加 CSV 讀取上限，避免大檔報錯
csv.field_size_limit(sys.maxsize)

131072

In [3]:
def print_stat(step_name, original_count, new_count, description=""):
    """ 格式化輸出統計數據 """
    diff = new_count - original_count
    ratio = (new_count / original_count * 100) if original_count > 0 else 0
    print(f"\n📊 [{step_name}] 統計報告:")
    print(f"   - 原始輸入 (Input): {original_count:,} 筆")
    print(f"   - 處理結果 (Output): {new_count:,} 筆")
    print(f"   - 變化 (Change): {diff:,} ({ratio:.1f}%)")
    if description:
        print(f"   - 備註: {description}")
    print("-" * 50)

In [4]:
# ==========================================
# Step 1: 建立臨床母群體 (Clinical Cohort)
# ==========================================
def step1_build_cohort():
    print("🚀 [Step 1] 讀取臨床核心表格...")
    
    # 讀取
    df_icu = pd.read_csv(PATH_ICUSTAYS)
    df_adm = pd.read_csv(PATH_ADMISSIONS)
    df_pat = pd.read_csv(PATH_PATIENTS)
    
    # 紀錄原始大小
    raw_icu_len = len(df_icu)
    print(f"   > ICUSTAYS 原檔: {raw_icu_len:,} 筆")
    print(f"   > ADMISSIONS 原檔: {len(df_adm):,} 筆")
    print(f"   > PATIENTS 原檔: {len(df_pat):,} 筆")

    # 欄位標準化
    for df in [df_icu, df_adm, df_pat]:
        df.columns = df.columns.str.lower().str.strip()

    # 串接
    df = df_icu.merge(df_adm[['subject_id', 'hadm_id', 'hospital_expire_flag']], on=['subject_id', 'hadm_id'], how='left')
    df = df.merge(df_pat[['subject_id', 'anchor_age', 'gender', 'dod']], on='subject_id', how='left')
    
    # 時間轉換
    for col in ['intime', 'outtime', 'dod']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # 處理死亡時間 (優先用 dod)
    df = df.rename(columns={'dod': 'deathtime'})

    # --- 篩選條件 ---
    # 1. 成年人 (Age >= 18)
    # 2. 病房: MICU, SICU, CCU, CSRU (排除 Neuro/Trauma)
    
    mask_adult = df['anchor_age'] >= 18
    
    # 定義病房關鍵字 (包含心臟科以極大化數據)
    target_units = 'MICU|Medical|SICU|Surgical|Coronary|Cardiac|CCU|CSRU'
    exclude_units = 'Neuro|Trauma'
    
    mask_unit = df['first_careunit'].str.contains(target_units, case=False, na=False) & \
               ~df['first_careunit'].str.contains(exclude_units, case=False, na=False)
    
    df_filtered = df[mask_adult & mask_unit].copy()
    
    print_stat("Step 1: Cohort Selection", raw_icu_len, len(df_filtered), 
               "篩選條件：成年人 + 內/外/心臟加護病房 (排除神經科)")
    
    return df_filtered

In [5]:
# ==========================================
# Step 2: 串接插管資料 (Intubation)
# ==========================================
def step2_merge_ventilation(df_cohort):
    print("\n🚀 [Step 2] 讀取處置紀錄 (Procedures)...")
    
    df_proc = pd.read_csv(PATH_PROCEDURES)
    raw_proc_len = len(df_proc)
    print(f"   > PROCEDUREEVENTS 原檔: {raw_proc_len:,} 筆")
    
    df_proc.columns = df_proc.columns.str.lower().str.strip()
    
    # 篩選插管 (Itemid 224385)
    vent = df_proc[df_proc['itemid'] == 224385].copy()
    vent['starttime'] = pd.to_datetime(vent['starttime'], errors='coerce')
    
    print(f"   > 插管事件 (Intubation Events): {len(vent):,} 筆")
    
    # 合併 (Inner Join)
    df_merged = df_cohort.merge(vent[['stay_id', 'starttime']], on='stay_id', how='inner')
    
    # 去重：每個人只取「第一次」插管
    df_merged = df_merged.sort_values('starttime')
    df_merged = df_merged.drop_duplicates(subset=['stay_id'], keep='first')
    df_merged = df_merged.rename(columns={'starttime': 'intubation_time'})
    
    print_stat("Step 2: Ventilation Merge", len(df_cohort), len(df_merged), 
               "只保留有插管紀錄的病患 (Inner Join)")
    
    return df_merged

In [6]:
# ==========================================
# Step 2.5: 讀取 X 光報告 (吸塵器模式)
# ==========================================
def step2_5_load_radiology(path_to_csv, target_ids):
    print("\n🚀 [Step 2.5] 掃描 X 光報告 (Vacuum Mode)...")
    target_set = set(target_ids)
    
    collected_rows = []
    header = []
    total_lines = 0
    saved_count = 0
    
    # 判斷壓縮檔
    if path_to_csv.endswith('.gz'):
        open_func = gzip.open; mode = 'rt'
    else:
        open_func = open; mode = 'r'

    try:
        with open_func(path_to_csv, mode, encoding='utf-8', errors='replace') as f:
            reader = csv.reader(f)
            try:
                header = next(reader)
                header = [h.lower().strip() for h in header]
                if 'subject_id' not in header: return pd.DataFrame()
                idx_subj = header.index('subject_id')
            except StopIteration:
                return pd.DataFrame()

            # 逐行掃描
            for row in reader:
                total_lines += 1
                if total_lines % 500000 == 0:
                    print(f"   ...已掃描 {total_lines:,} 行 (捕獲 {saved_count:,} 筆)")
                
                try:
                    if len(row) <= idx_subj: continue
                    curr_id = row[idx_subj]
                    if not curr_id.isdigit(): continue
                    
                    # 只要 ID 在名單內就抓 (不做文字快篩，避免漏抓)
                    if int(curr_id) in target_set:
                        collected_rows.append(row)
                        saved_count += 1
                except:
                    continue
                    
    except Exception as e:
        print(f"   ⚠️ 讀取中斷 (正常現象): {e}")

    df_rad = pd.DataFrame(collected_rows, columns=header)
    
    print_stat("Step 2.5: Radiology Scan", total_lines, saved_count, 
               "從原始大檔中撈取目標病患的所有報告")
    
    return df_rad

In [7]:
# ==========================================
# Step 3: 時間配對 (入院時間錨點)
# ==========================================
def step3_match_cxr(df_cohort, df_rad):
    print("\n🚀 [Step 3] 執行時間配對 (Broad Window)...")
    
    # 紀錄進入 Step 3 的人數
    cohort_count = len(df_cohort)
    
    # 1. 欄位處理
    df_rad.columns = df_rad.columns.str.lower().str.strip()
    df_rad['subject_id'] = pd.to_numeric(df_rad['subject_id'], errors='coerce')
    df_rad = df_rad.dropna(subset=['subject_id'])
    df_rad['subject_id'] = df_rad['subject_id'].astype('int64')
    
    df_cohort['subject_id'] = df_cohort['subject_id'].astype('int64')

    # 2. 時間轉換 (移除時區)
    df_rad['charttime'] = pd.to_datetime(df_rad['charttime'], errors='coerce').dt.tz_localize(None)
    df_rad = df_rad.dropna(subset=['charttime'])
    
    df_cohort['intime'] = pd.to_datetime(df_cohort['intime'], errors='coerce').dt.tz_localize(None)

    # 3. 合併
    id_col = 'note_id' if 'note_id' in df_rad.columns else 'report_id'
    cols = ['subject_id', 'charttime', 'text', id_col]
    cols = [c for c in cols if c in df_rad.columns] # 防呆
    
    df_merged = df_cohort.merge(df_rad[cols], on='subject_id', how='left')
    
    # 4. 篩選時間窗口: ICU 入院前 24h ~ 入院後 72h
    # 這能抓到急診插管 (Pre-ICU) 的片子
    mask_cxr = df_merged['charttime'].notna()
    mask_time = (df_merged['charttime'] >= (df_merged['intime'] - pd.Timedelta(hours=24))) & \
                (df_merged['charttime'] <= (df_merged['intime'] + pd.Timedelta(hours=72)))
    
    valid_cxr = df_merged[mask_cxr & mask_time].copy()
    
    # 5. 取最接近 ICU 入院的一張
    valid_cxr['time_diff'] = (valid_cxr['charttime'] - valid_cxr['intime']).abs()
    valid_cxr = valid_cxr.sort_values(['subject_id', 'time_diff'])
    valid_cxr = valid_cxr.drop_duplicates(subset=['stay_id'], keep='first')
    
    print_stat("Step 3: Time Matching", cohort_count, len(valid_cxr), 
               "配對 ICU 入院前後 (-24h~+72h) 的 X 光報告")
    
    return valid_cxr

In [8]:
# ==========================================
# Step 4: 臨床排除標準
# ==========================================
def step4_apply_exclusions(df):
    print("\n🚀 [Step 4] 執行臨床排除...")
    original_n = len(df)
    
    # 1. 排除住院 < 1 天
    df['outtime'] = pd.to_datetime(df['outtime'], errors='coerce').dt.tz_localize(None)
    df['intime'] = pd.to_datetime(df['intime'], errors='coerce').dt.tz_localize(None)
    df['icu_los_days'] = (df['outtime'] - df['intime']).dt.total_seconds() / 86400
    
    df_los = df[df['icu_los_days'] >= 1].copy()
    print(f"   - 排除極短住院 (<1天): 剔除 {len(df) - len(df_los)} 筆")
    
    # 2. 排除氣切 (Tracheostomy)
    mask_trach = df_los['text'].str.lower().str.contains('tracheostomy|trach tube', na=False)
    df_final = df_los[~mask_trach].copy()
    print(f"   - 排除氣切關鍵字: 剔除 {len(df_los) - len(df_final)} 筆")
    
    print_stat("Step 4: Exclusions", original_n, len(df_final), 
               "排除 LOS<1天 與 氣切病患")
    
    return df_final

In [9]:
# ==========================================
# 主程式 (Main)
# ==========================================
if __name__ == "__main__":
    if not os.path.exists(PATH_RADIOLOGY):
        print(f"❌ 錯誤：找不到 Radiology 檔案: {PATH_RADIOLOGY}")
    else:
        # Pipeline Execution
        df_cohort = step1_build_cohort()
        df_vent = step2_merge_ventilation(df_cohort)
        
        target_ids = df_vent['subject_id'].unique()
        df_rad_full = step2_5_load_radiology(PATH_RADIOLOGY, target_ids)
        
        if not df_rad_full.empty:
            df_matched = step3_match_cxr(df_vent, df_rad_full)
            df_final = step4_apply_exclusions(df_matched)
            
            # 最終存檔
            output_file = 'final_mimic_dataset.csv'
            df_final.to_csv(output_file, index=False)
            print(f"\n✅✅✅ 處理完成！檔案已儲存至: {output_file}")
            print(f"最終有效樣本數: {len(df_final):,} 筆")
        else:
            print("❌ 警告：沒有抓到任何 X 光報告，請檢查 ID 是否匹配。")

🚀 [Step 1] 讀取臨床核心表格...
   > ICUSTAYS 原檔: 94,458 筆
   > ADMISSIONS 原檔: 546,028 筆
   > PATIENTS 原檔: 364,627 筆

📊 [Step 1: Cohort Selection] 統計報告:
   - 原始輸入 (Input): 94,458 筆
   - 處理結果 (Output): 74,707 筆
   - 變化 (Change): -19,751 (79.1%)
   - 備註: 篩選條件：成年人 + 內/外/心臟加護病房 (排除神經科)
--------------------------------------------------

🚀 [Step 2] 讀取處置紀錄 (Procedures)...
   > PROCEDUREEVENTS 原檔: 808,706 筆
   > 插管事件 (Intubation Events): 9,777 筆

📊 [Step 2: Ventilation Merge] 統計報告:
   - 原始輸入 (Input): 74,707 筆
   - 處理結果 (Output): 6,986 筆
   - 變化 (Change): -67,721 (9.4%)
   - 備註: 只保留有插管紀錄的病患 (Inner Join)
--------------------------------------------------

🚀 [Step 2.5] 掃描 X 光報告 (Vacuum Mode)...
   ...已掃描 500,000 行 (捕獲 46,755 筆)
   ...已掃描 1,000,000 行 (捕獲 96,710 筆)
   ...已掃描 1,500,000 行 (捕獲 144,102 筆)
   ...已掃描 2,000,000 行 (捕獲 189,425 筆)

📊 [Step 2.5: Radiology Scan] 統計報告:
   - 原始輸入 (Input): 2,321,355 筆
   - 處理結果 (Output): 219,612 筆
   - 變化 (Change): -2,101,743 (9.5%)
   - 備註: 從原始大檔中撈取目標病患的所有報告
-----------